# 삼성디스플레이 AI Agent Architecture 과정
# 2일차 LangGraph 강사용 노트북
## RAG Agent v1의 StateGraph, Node, 조건부 분기, Trace 설계

---

이 노트북은 2일차 전체 RAG 실습을 대체하는 자료가 **아닙니다**.
이 노트북은 RAG Agent v1의 실행 흐름을 **LangGraph StateGraph**로 구성하고,
**State, Node, 조건부 분기, Trace, 그래프 시각화**를 통해 Agent 실행 구조를 설계 관점에서 이해하기 위한 강사용 보조 자료입니다.

- 실제 프로젝트 기준 파일은 `src/day2` 폴더의 Python 파일입니다.
- 이 노트북은 **LangGraph 구조만 집중적으로** 설명합니다.

**핵심 메시지**
LangGraph는 단순 실행 순서 도구가 아니라,
Agent의 **State, Node, 조건부 분기, Trace를 명시적으로 설계하기 위한 구조**입니다.

## 강의 환경 기준 (uv 가상환경 / LangGraph 1.2.2)

이 노트북은 프로젝트의 **uv 기반 가상환경**과 **`langgraph==1.2.2`** 기준으로 작성되었습니다.
LangGraph는 업데이트가 빠른 라이브러리이므로, 강의에서는 검증된 버전 기준으로 진행하는 것이 안전합니다.

> 참고용 명령(노트북 코드 셀에서 실행하지 않습니다. 터미널에서 직접 확인):
>
> ```powershell
> uv pip list | findstr lang
> uv run python -c "from langgraph.graph import StateGraph, END; print('LangGraph import ok')"
> ```
>
> 새 패키지를 설치하지 않습니다(`uv add` / `uv pip install` / `pip install` 사용 금지). 현재 설치된 환경 기준으로 진행합니다.

기본 import는 `from langgraph.graph import StateGraph, END` 형태를 사용하며,
LangGraph가 없거나 import가 실패해도 노트북이 중단되지 않도록 안전하게 처리합니다.

## 사용할 데이터

| 파일 | LangGraph 노트북에서의 역할 |
|---|---|
| `data/sample_query.json` | initial_state 생성용 대표 사용자 질의 |
| `data/sample_alarm_logs.csv` | 로그 검색 Node의 근거 데이터 |
| `data/sample_rag_queries.json` | RAG 검색 질의와 query rewrite 예시 |
| `data/tool_selection_test_cases.json` | 3일차 Tool/MCP 연결 예고 (선택) |
| `data/action_lab_scenarios.json` | 5일차 Action Lab 확장 예고 (선택) |

모든 데이터는 실제 사내 데이터가 아니라 **교육용 가상 데이터**이며, LangGraph State와 Node 흐름을 설명하기 위한 입력입니다.
파일이 없어도 노트북이 중단되지 않도록, 각 단계에서 교육용 기본 데이터(fallback)를 코드 안에서 사용합니다.

## 0. 준비: 프로젝트 루트와 공통 유틸리티

노트북 위치에 상관없이 **프로젝트 루트를 자동 탐색**하고(경로 하드코딩 금지),
입력 데이터 확인·Markdown 저장·fallback 데이터 제공을 돕는 함수를 정의합니다.
파일이 없어도 traceback 대신 안내 메시지로 대체해 **Run All이 중단되지 않게** 합니다.

In [ ]:
# 이 셀은 LangGraph 전용 노트북 전체에서 사용할 공통 유틸리티를 준비하는 셀입니다.
# - find_project_root: 특정 절대 경로에 의존하지 않도록 프로젝트 루트를 자동 탐색합니다(경로 하드코딩 방지).
# - load_json / load_csv_rows / preview_* : 입력 데이터와 산출물을 강의 중 빠르게 미리보기하기 위한 도구입니다.
# - _missing 및 각 Node의 FALLBACK 데이터: 파일 위치가 달라도 강의 흐름이 끊기지 않도록 교육용 fallback으로 대체합니다.
# - save_markdown: LangGraph 실행 결과와 Trace를 outputs/day2 기준 산출물로 남기기 위한 함수입니다.
# - esc: Markdown 표 셀이 깨지지 않도록 줄바꿈/파이프 문자를 치환합니다.
from pathlib import Path
import json
import csv
import re
from IPython.display import Markdown, display


def find_project_root(markers=("src", "data", "outputs")):
    """현재 폴더에서 상위로 올라가며 src/data/outputs 중 2개 이상이 있는 위치를 루트로 판단합니다."""
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        hits = sum(1 for name in markers if (base / name).is_dir())
        if hits >= 2:
            return base
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    print("프로젝트 루트를 찾지 못했습니다. 현재 폴더:", Path.cwd())
    print("src / data / outputs 폴더가 있는 위치(프로젝트 루트)에서 실행해 주세요.")
else:
    print("프로젝트 루트:", PROJECT_ROOT)


def _resolve(rel):
    return None if PROJECT_ROOT is None else (PROJECT_ROOT / rel)


def _missing(rel):
    display(Markdown(f"> `{rel}` 파일이 없어 교육용 fallback 데이터로 진행합니다."))


def esc(value):
    """Markdown 표 셀이 깨지지 않도록 줄바꿈/파이프를 치환합니다."""
    return str(value if value is not None else "").replace("\n", " ").replace("|", "/")


def show_markdown(rel):
    path = _resolve(rel)
    if path is None or not path.exists():
        _missing(rel)
        return
    display(Markdown(path.read_text(encoding="utf-8-sig")))


def load_json(rel):
    """JSON 파일을 읽어 객체로 반환합니다. 없으면 None."""
    path = _resolve(rel)
    if path is None or not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8-sig"))


def preview_json(rel, limit=3, fields=None, list_key=None):
    """JSON 산출물의 앞부분 일부 항목만 출력합니다(전체 출력 금지)."""
    data = load_json(rel)
    if data is None:
        _missing(rel)
        return
    items = data
    if list_key and isinstance(data, dict):
        print("최상위 키:", list(data.keys()))
        items = data.get(list_key, [])
    if isinstance(items, list):
        print(f"총 {len(items)}개 항목 중 앞 {min(limit, len(items))}개만 표시합니다.")
        preview = items[:limit]
        if fields:
            preview = [{key: it.get(key, "") for key in fields} for it in preview]
        for it in preview:
            display(it)
    elif isinstance(items, dict) and fields:
        display({key: items.get(key, "") for key in fields if key in items})
    else:
        display(items)


def load_csv_rows(rel):
    """CSV를 dict 행 리스트로 읽습니다. 없으면 빈 리스트."""
    path = _resolve(rel)
    if path is None or not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def preview_csv(rel, n=5, fields=None):
    """CSV 파일의 앞 n행만 미리보기합니다."""
    rows = load_csv_rows(rel)
    if not rows:
        _missing(rel)
        return
    print(f"[{rel}] 총 {len(rows)}행 중 앞 {min(n, len(rows))}행만 표시합니다.")
    for row in rows[:n]:
        display({k: row.get(k, "") for k in (fields or row.keys())})


def save_markdown(rel, text):
    path = _resolve(rel)
    if path is None:
        print("프로젝트 루트를 찾지 못해 저장을 건너뜁니다:", rel)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8-sig")
    print("저장 완료:", rel)


def extract_equipment_id(text):
    m = re.search(r"EQP-[A-Z]+-[0-9]+", text or "")
    return m.group(0) if m else ""


def extract_alarm_code(text):
    m = re.search(r"ALM-[A-Z]+-[0-9]+", text or "")
    return m.group(0) if m else ""

## 1. LangGraph 버전 확인

LangGraph 관련 패키지의 설치 버전을 확인합니다.
강의 기준 버전은 **`langgraph==1.2.2`** 입니다. 설치되어 있지 않은 패키지가 있어도 노트북은 중단되지 않습니다.

In [ ]:
# 이 셀은 현재 실습 환경(uv 가상환경)의 LangGraph 관련 패키지 버전을 확인하는 셀입니다.
# - LangGraph는 업데이트가 빠른 라이브러리이므로, 강의는 검증된 기준 버전(langgraph==1.2.2)으로 진행합니다.
# - 설치되어 있지 않은 패키지가 있어도 노트북이 중단되지 않도록 처리합니다.
# - 버전 확인은 이후 단계에서 '코드 오류'와 '환경 오류'를 구분하는 데 도움이 됩니다.
import importlib.metadata as _meta

_packages = ["langgraph", "langchain", "langchain-core",
             "langgraph-checkpoint", "langgraph-prebuilt", "langgraph-sdk", "langsmith"]

print("현재 환경의 LangGraph 관련 패키지 버전")
print("-" * 45)
for name in _packages:
    try:
        print(f"{name:24s} {_meta.version(name)}")
    except Exception:
        print(f"{name:24s} (설치되어 있지 않음)")
print("-" * 45)
print("강의 기준 버전: langgraph==1.2.2")

## 2. 입력 데이터 미리보기

LangGraph State와 Node 흐름의 입력이 되는 교육용 가상 데이터를 **일부만** 확인합니다.

In [ ]:
# sample_query.json: initial_state 생성에 사용할 대표 사용자 질의입니다(주요 필드만 미리보기).
# - 실제 사내 데이터가 아니라 교육용 가상 데이터이며, 이후 build_initial_state의 입력이 됩니다.
preview_json("data/sample_query.json",
             fields=["scenario_name", "user_query", "line_id", "process_name", "equipment_id", "alarm_code"])

In [ ]:
# sample_alarm_logs.csv: retrieve_alarm_logs_node가 조회할 로그 기반 근거 데이터입니다(앞 5행).
# - 반복 발생 여부·심각도·반복 횟수·현장 메모를 판단하는 교육용 가상 근거 데이터입니다.
preview_csv("data/sample_alarm_logs.csv", n=5,
            fields=["timestamp", "equipment_id", "alarm_code", "severity", "repeat_count", "operator_note"])

In [ ]:
# sample_rag_queries.json: retrieve_docs_node와 query_rewrite_node 설명에 사용할 샘플 RAG 질의입니다(앞 3개).
# - expected_keywords/expected_docs는 문서 근거 후보를 구성하는 교육용 가상 데이터입니다.
preview_json("data/sample_rag_queries.json", limit=3, list_key="queries",
             fields=["id", "user_query", "intent", "difficulty"])

In [ ]:
# (선택) 3일차/5일차 확장 예고용 데이터 미리보기입니다(앞 2개만).
# - tool_selection_test_cases.json은 3일차 MCP Tool 선택 예고, action_lab_scenarios.json은 5일차 Action Lab 확장 예고에 연결됩니다.
# - 모두 실제 사내 데이터가 아니라 교육용 가상 데이터입니다.
print("== tool_selection_test_cases.json (3일차 Tool/MCP 예고) ==")
preview_json("data/tool_selection_test_cases.json", limit=2,
             fields=["case_id", "user_query", "expected_tools"])
print()
print("== action_lab_scenarios.json (5일차 Action Lab 예고) ==")
preview_json("data/action_lab_scenarios.json", limit=2,
             fields=["scenario_id", "action_name", "user_query"])

## 3. State 구조 정의 (AgentState)

State는 **Node 사이에서 공유되는 업무 처리 문맥**입니다.
단순 변수 묶음이 아니라, Agent가 **어떤 근거를 검색했고(retrieved_logs/retrieved_docs), 어떤 답변을 만들었고(draft/final_answer),
어떤 검증을 거쳤는지(grounding/helpfulness)** 를 다음 Node로 전달하는 구조입니다.

In [ ]:
# 이 셀은 StateGraph 전체에서 공유되는 State 스키마(AgentState)를 정의하는 셀입니다.
# - State는 Node 사이를 이동하는 '업무 처리 문맥'으로, 어떤 근거를 검색했고 어떤 답변을 만들었으며
#   어떤 검증/보정을 거쳤는지를 다음 Node로 전달합니다.
# - 아래 필드 주석은 강의 중 각 값이 어느 Node에서 채워지고 어디서 쓰이는지를 설명하기 위한 것입니다.
# AgentState: StateGraph 전체에서 공유되는 State 스키마
from typing import TypedDict, List, Dict, Any


class AgentState(TypedDict):
    user_query: str            # 사용자 질문 = 실행 흐름의 원본 입력
    rewritten_query: str       # 근거 부족 시 재검색용으로 다시 쓴 질의
    equipment_id: str          # 질문/시나리오에서 확인한 설비 ID(검색·Tool 조건)
    alarm_code: str            # 질문/시나리오에서 확인한 알람 코드(검색·Tool 조건)
    retrieved_docs: List[Dict[str, Any]]  # 문서 기반 근거 후보(Top-3) = 답변·grounding 입력
    retrieved_logs: List[Dict[str, Any]]  # 로그 기반 근거 = 반복 발생/심각도 판단 입력
    draft_answer: str          # 검증 전 답변 초안
    final_answer: str          # 검증까지 거친 최종 답변
    grounding_status: str      # 근거 반영 여부(not_checked / grounded / insufficient)
    helpfulness_status: str    # 답변 유용성 판정(not_checked / helpful / needs_improvement)
    needs_rewrite: bool        # 조건부 분기 기준(True면 질의 재작성 경로)
    retry_count: int           # 재검색 시도 횟수(무한 재작성 방지 상한 비교)
    errors: List[str]          # 실행 중 누적 오류(품질 검토·디버깅 근거)
    trace: List[Dict[str, str]]  # Node 실행 순서와 State 변화 이력 = 4일차 품질 평가 기반
    # --- Self-RAG / Corrective RAG 보강 필드 ---
    self_rag_review: Dict[str, Any]        # Self-RAG식 검증 결과(문서 관련성/근거성/유용성/단정/민감정보)
    corrective_plan: List[Dict[str, Any]]  # 근거 부족 시 제조 Tool/MCP fallback 보정 '계획'(실제 호출 아님)
    recommended_tools: List[str]           # 3일차 MCP Tool로 확장 가능한 Tool 후보(중복 없음)
    guardrail_warnings: List[str]          # 원인 단정·근거 부족·민감정보 가능성 등 안전 경고


def add_trace(state, node_name, status, message, input_summary="", output_summary=""):
    """Node 실행 기록을 trace에 한 줄로 남깁니다. trace는 4일차 실행 품질 평가의 기준 자료입니다."""
    state["trace"].append({"node_name": node_name, "status": status, "message": message,
                           "input_summary": input_summary, "output_summary": output_summary})
    return state


print("AgentState 필드:", list(AgentState.__annotations__.keys()))

## 4. initial_state 생성

`data/sample_query.json`을 기반으로 초기 State를 만듭니다.
파일이 없으면 교육용 fallback 질의를 사용합니다.

In [ ]:
# 이 셀은 LangGraph 실행의 시작 상태(initial_state)를 만드는 셀입니다.
# - sample_query.json이 있으면 교육용 대표 질의를, 없으면 fallback 질의를 사용해 강의 흐름을 유지합니다.
# - 검색 결과/답변/검증 결과/Trace는 비어 있는 상태로 시작하며, 이 State가 parse_query_node로 전달되어 Graph 실행이 시작됩니다.
# sample_query.json 기반 initial_state 생성 (없으면 fallback 질의 사용)
FALLBACK_QUERY = ("EQP-EV-03에서 ALM-TEMP-402 알람이 반복 발생했습니다. "
                  "반복 발생 여부, 원인 후보, 1차 확인 항목, 추가 확인 필요 사항을 정리해 주세요.")


def build_initial_state():
    """sample_query.json의 user_query/equipment_id/alarm_code를 우선 사용해 초기 State를 만듭니다."""
    data = load_json("data/sample_query.json") or {}
    user_query = data.get("user_query") or FALLBACK_QUERY
    # 시나리오 파일에 명시된 값이 있으면 우선, 없으면 질문 문장에서 추출
    equipment_id = data.get("equipment_id") or extract_equipment_id(user_query)
    alarm_code = data.get("alarm_code") or extract_alarm_code(user_query)
    return {
        "user_query": user_query,
        "rewritten_query": "",
        "equipment_id": equipment_id,
        "alarm_code": alarm_code,
        "retrieved_docs": [],
        "retrieved_logs": [],
        "draft_answer": "",
        "final_answer": "",
        "grounding_status": "not_checked",
        "helpfulness_status": "not_checked",
        "needs_rewrite": False,
        "retry_count": 0,
        "errors": [],
        "trace": [],
        # Self-RAG / Corrective RAG 보강 필드 기본값(없을 때도 안전하게 빈 dict/list)
        "self_rag_review": {},
        "corrective_plan": [],
        "recommended_tools": [],
        "guardrail_warnings": [],
    }


INITIAL_STATE = build_initial_state()
print("user_query   :", INITIAL_STATE["user_query"])
print("equipment_id :", INITIAL_STATE["equipment_id"])
print("alarm_code   :", INITIAL_STATE["alarm_code"])

## 5. Node 함수 정의

각 Node는 **State를 입력받아 필요한 값을 채운 뒤 다음 Node로 넘기는 Agent 실행 단위**입니다.
외부 LLM/API/웹 검색은 호출하지 않으며, 모든 근거는 교육용 가상 데이터에서 가져옵니다.

In [ ]:
# 이 셀은 RAG Agent v1의 Node와 Self-RAG/Corrective RAG 보조 함수를 정의하는 가장 핵심적인 셀입니다.
# - 각 Node는 State를 입력받아 필요한 값을 채운 뒤 다음 Node로 넘기는 'Agent 실행 단위'입니다.
# - 외부 LLM/API/웹 검색은 호출하지 않으며, 모든 근거는 교육용 가상 데이터에서 가져옵니다.
# - run_self_rag_review는 답변 검증(Self-RAG), build_corrective_plan/corrective_rag_node는 근거 부족 시
#   제조 Tool/MCP fallback 보정 계획을 만드는 함수로, 실제 Tool 호출은 하지 않습니다.
# Node 1: parse_query_node — 질문에서 검색·Tool 호출 조건(설비 ID/알람 코드)을 확정합니다.
def parse_query_node(state):
    # sample_query.json에 값이 있으면 그 값을 우선 사용하고, 비어 있으면 질문에서 추출합니다.
    query = state.get("rewritten_query") or state.get("user_query", "")
    if not state.get("equipment_id"):
        state["equipment_id"] = extract_equipment_id(query)
    if not state.get("alarm_code"):
        state["alarm_code"] = extract_alarm_code(query)
    add_trace(state, "parse_query_node", "success", "검색·Tool 호출 조건(설비 ID/알람 코드)을 확정했습니다.",
              query, f"equipment_id={state['equipment_id'] or '없음'}, alarm_code={state['alarm_code'] or '없음'}")
    return state


# Node 2: retrieve_alarm_logs_node — 로그 기반 근거(반복 발생/심각도 판단의 입력)를 확보합니다.
FALLBACK_LOGS = [
    {"timestamp": "2026-05-01 09:05:12", "equipment_id": "EQP-EV-03", "alarm_code": "ALM-TEMP-402",
     "severity": "WARNING", "repeat_count": "1", "alarm_message": "Training chamber temperature deviation",
     "operator_note": "챔버 온도 편차 알람 최초 확인"},
    {"timestamp": "2026-05-01 10:06:31", "equipment_id": "EQP-EV-03", "alarm_code": "ALM-TEMP-402",
     "severity": "CRITICAL", "repeat_count": "3", "alarm_message": "Training repeated chamber temperature deviation",
     "operator_note": "온도 편차가 반복되어 확인 필요"},
]


def retrieve_alarm_logs_node(state):
    # retrieved_logs는 반복 발생 여부와 심각도를 판단하는 로그 기반 근거입니다.
    rows = load_csv_rows("data/sample_alarm_logs.csv")
    if not rows:
        rows = FALLBACK_LOGS  # 로그 파일이 없을 때 교육용 fallback 로그로 흐름을 유지합니다.
    equip = state.get("equipment_id", "")
    alarm = state.get("alarm_code", "")
    keep = ("timestamp", "severity", "repeat_count", "alarm_message", "operator_note",
            "equipment_id", "alarm_code", "chamber_id")
    logs = []
    for row in rows:
        # 설비 ID/알람 코드 조건이 있으면 그에 맞는 로그만 근거로 채택합니다.
        if equip and row.get("equipment_id", "") != equip:
            continue
        if alarm and row.get("alarm_code", "") != alarm:
            continue
        logs.append({k: row.get(k, "") for k in keep if k in row})
    if not logs:  # 조건과 정확히 일치하는 로그가 없으면 앞쪽 로그를 참고 근거로 사용합니다.
        logs = [{k: r.get(k, "") for k in keep if k in r} for r in rows[:3]]
    state["retrieved_logs"] = logs
    add_trace(state, "retrieve_alarm_logs_node", "success" if logs else "warning",
              "설비 ID/알람 코드 기준으로 관련 로그를 필터링했습니다.",
              f"equipment_id={equip}, alarm_code={alarm}", f"retrieved_logs={len(logs)}건")
    return state


# Node 3: retrieve_docs_node — 문서 기반 근거 후보(Top-3)를 확보합니다.
FALLBACK_DOCS = [
    {"rank": 1, "score": 0.78, "doc_name": "alarm_manual.md", "section_title": "ALM-TEMP-402 알람 의미",
     "text": "ALM-TEMP-402는 챔버 온도 편차 관련 교육용 알람으로, 반복 발생 여부를 함께 확인해야 합니다.",
     "keywords": "ALM-TEMP-402, 온도 편차, 반복 발생"},
    {"rank": 2, "score": 0.71, "doc_name": "troubleshooting_guide.md", "section_title": "온도 편차 1차 확인 항목",
     "text": "반복 발생 시 동일 챔버/설비/시각 집중 여부와 현장 메모를 함께 확인합니다.",
     "keywords": "1차 확인, 반복 발생, 현장 메모"},
]


def retrieve_docs_node(state):
    # 문서형 RAG 본문 파일이 없을 수 있으므로 sample_rag_queries.json을 근거 후보 원천으로 사용합니다.
    query = state.get("rewritten_query") or state.get("user_query", "")
    data = load_json("data/sample_rag_queries.json")
    queries = (data.get("queries", []) if isinstance(data, dict) else data) or []
    terms = [w for w in re.split(r"\s+", query.strip()) if len(w) >= 2]

    candidates = []
    for q in queries:
        keywords = q.get("expected_keywords", []) or []
        docs = q.get("expected_docs", []) or []
        text = str(q.get("user_query", "") or "")
        hay = text + " " + " ".join(keywords)
        # 질문 토큰이 얼마나 겹치는지로 근거 후보 점수를 계산합니다(외부 LLM 미사용).
        hits = sum(1 for t in terms if t in hay)
        if hits <= 0:
            continue
        candidates.append({
            "score": round(hits / max(len(terms), 1), 4),
            "doc_name": docs[0] if docs else "sample_rag_queries.json",
            "section_title": str(q.get("intent", "") or q.get("id", "")),
            "text": text,
            "keywords": ", ".join(str(k) for k in keywords),
        })
    candidates.sort(key=lambda x: x["score"], reverse=True)
    top = candidates[:3]
    for i, c in enumerate(top):
        c["rank"] = i + 1
    if not top:
        top = FALLBACK_DOCS  # 매칭 결과가 없으면 교육용 fallback 근거 후보를 사용합니다.
    state["retrieved_docs"] = top
    add_trace(state, "retrieve_docs_node", "success" if top else "warning",
              "문서 기반 Top-3 근거 후보를 확보했습니다(정답이 아니라 검토할 후보 집합).",
              query, f"retrieved_docs={len(top)}건")
    return state


# Node 4: generate_answer_node — 검색 근거 기반 교육용 답변 초안을 만듭니다(외부 LLM 미호출).
def generate_answer_node(state):
    logs = state.get("retrieved_logs", [])
    docs = state.get("retrieved_docs", [])
    severities = [str(r.get("severity", "")) for r in logs]
    has_critical = any(s == "CRITICAL" for s in severities)
    repeat_values = [r.get("repeat_count", "") for r in logs]

    log_line = (f"- 관련 로그 {len(logs)}건 확인(심각도: {', '.join(s for s in severities if s) or '정보 없음'}, "
                f"repeat_count: {', '.join(str(v) for v in repeat_values if v) or '정보 없음'})")
    doc_line = "\n".join(f"- 근거 후보: {d.get('doc_name', '')} / {d.get('section_title', '')} (score={d.get('score', '')})"
                         for d in docs) or "- 문서 근거 후보가 없습니다."

    # 원인을 단정하지 않고 원인 후보 / 1차 확인 항목 / 추가 확인 필요 사항 중심으로 작성합니다.
    answer = (
        "## 교육용 답변 초안\n"
        f"- 설비 ID: {state.get('equipment_id', '') or '미확인'}\n"
        f"- 알람 코드: {state.get('alarm_code', '') or '미확인'}\n\n"
        "### 로그·문서 근거\n"
        f"{log_line}\n{doc_line}\n\n"
        "### 반복 발생 여부\n"
        f"- 동일 설비/알람 코드 기준으로 반복 발생 여부를 확인했습니다."
        f"{' (CRITICAL 포함, 반복성 주의)' if has_critical else ''}\n\n"
        "### 원인 후보\n"
        "- 챔버 온도 제어 상태 확인 필요\n- 센서 값 변동 가능성 확인 필요\n- 인접 알람(진공/압력/가스 흐름)과의 연관 가능성 확인 필요\n\n"
        "### 1차 확인 항목\n"
        "- 동일 챔버/시각 집중 발생 여부\n- severity와 repeat_count 추이\n- 현장 메모(operator_note)의 재발/확인 필요 표현\n\n"
        "### 추가 확인 필요 사항\n"
        "- 원인은 확정하지 않으며, 품질 영향 여부는 별도 품질 데이터 확인이 필요합니다.\n"
        "- 추가 점검과 담당자 검토가 필요한 후보로만 정리합니다."
    )
    state["draft_answer"] = answer
    state["final_answer"] = answer
    add_trace(state, "generate_answer_node", "success",
              "로그·문서 근거 기반 교육용 답변 초안을 생성했습니다(외부 LLM 미호출).",
              f"logs={len(logs)}건, docs={len(docs)}건", "answer_source=notebook_draft")
    return state


# Node 5 보조 함수: run_self_rag_review — 외부 LLM grader 없이 '규칙 기반'으로 답변을 검증합니다.
def run_self_rag_review(state):
    # 이 함수는 2.4 Self-RAG의 답변 검증 아이디어를 외부 LLM 없이 규칙 기반으로 축소 적용한 것입니다.
    # - 실제 운영에서는 LLM grader, 평가 모델, 정책 기반 Guardrail로 확장할 수 있습니다.
    # - 2일차 노트북에서는 교육용으로 근거 존재, 답변 근거성, 유용성, 단정 표현 여부, 민감정보 가능성만 점검합니다.
    docs = state.get("retrieved_docs", []) or []
    logs = state.get("retrieved_logs", []) or []
    answer = state.get("draft_answer", "") or ""
    equip = state.get("equipment_id", "") or ""
    alarm = state.get("alarm_code", "") or ""

    review = {}
    warnings = []

    # 1) document_relevance: 검색 근거(문서/로그)가 하나라도 존재하는지 확인합니다.
    if docs or logs:
        review["document_relevance"] = {"status": "pass",
            "reason": f"문서 {len(docs)}건 / 로그 {len(logs)}건의 검색 근거가 존재합니다."}
    else:
        review["document_relevance"] = {"status": "fail",
            "reason": "retrieved_docs / retrieved_logs가 비어 있어 검색 근거가 없습니다."}

    # 2) answer_grounding: 답변이 설비 ID/알람 코드를 반영하고, 근거 키워드가 답변에 나타나는지 확인합니다.
    reflects_condition = (bool(equip) and equip in answer) or (bool(alarm) and alarm in answer)
    evidence_keywords = []
    for d in docs:
        evidence_keywords += [k.strip() for k in str(d.get("keywords", "")).split(",") if k.strip()]
    for lg in logs:
        if lg.get("alarm_code"):
            evidence_keywords.append(str(lg.get("alarm_code")))
    keyword_hit = any(k in answer for k in evidence_keywords) if evidence_keywords else False
    grounded_ok = reflects_condition or keyword_hit
    review["answer_grounding"] = {"status": "pass" if grounded_ok else "fail",
        "reason": ("답변이 설비 ID/알람 코드 또는 검색 근거 키워드를 반영합니다."
                   if grounded_ok else "답변에 설비 ID/알람 코드/근거 키워드가 반영되지 않았습니다.")}

    # 3) answer_helpfulness: 확인/점검/검토/조치/추가 확인 같은 실무 안내 표현이 포함되는지 확인합니다.
    practical_terms = ["확인", "점검", "검토", "조치", "추가 확인"]
    has_practical = any(t in answer for t in practical_terms)
    review["answer_helpfulness"] = {"status": "pass" if has_practical else "fail",
        "reason": ("확인/점검/검토/조치/추가 확인 등 실무 안내 표현이 포함됩니다."
                   if has_practical else "실무적 안내(확인/점검/검토/조치) 표현이 부족합니다.")}

    # 4) overclaim_check: 원인을 단정하는 표현이 있으면 warning. guardrail_warnings의 근거가 됩니다.
    overclaim_terms = ["확정 원인", "반드시", "즉시 조치", "무조건"]
    found_overclaim = [t for t in overclaim_terms if t in answer]
    if found_overclaim:
        review["overclaim_check"] = {"status": "warning",
            "reason": "단정 표현이 발견되었습니다: " + ", ".join(found_overclaim)}
        warnings.append("원인 단정 가능성: 단정 표현(" + ", ".join(found_overclaim) + ")이 답변에 포함되어 있습니다.")
    else:
        review["overclaim_check"] = {"status": "pass",
            "reason": "원인을 단정하는 표현이 발견되지 않았습니다."}

    # 5) safety_check: 개인정보/작업자 이름/사번/연락처 등 민감정보 가능성을 간단히 점검합니다.
    sensitive_terms = ["사번", "주민등록", "주민번호", "연락처", "전화번호", "휴대폰", "이메일", "@"]
    found_sensitive = [t for t in sensitive_terms if t in answer]
    if found_sensitive:
        review["safety_check"] = {"status": "warning",
            "reason": "민감정보 가능성 표현이 발견되었습니다: " + ", ".join(found_sensitive)}
        warnings.append("민감정보 가능성: 개인정보/연락처 등으로 보이는 표현이 답변에 포함될 수 있습니다.")
    else:
        review["safety_check"] = {"status": "pass",
            "reason": "개인정보·작업자 이름·사번·연락처 등 민감정보 표현이 발견되지 않았습니다."}

    return review, warnings


# Node 5: verify_grounding_node — run_self_rag_review로 Self-RAG식 검증을 수행하고 결과를 State에 반영하는 품질 관문입니다.
def verify_grounding_node(state):
    # 검증 로직 자체는 run_self_rag_review로 분리하고, 이 Node는 '검증 결과 -> 라우팅 판정' 책임만 맡습니다.
    review, warnings = run_self_rag_review(state)
    state["self_rag_review"] = review

    # 단정 표현·민감정보 가능성 경고는 guardrail_warnings에 누적합니다(중복 없이).
    gw = state.get("guardrail_warnings", []) or []
    for w in warnings:
        if w not in gw:
            gw.append(w)
    state["guardrail_warnings"] = gw

    # 핵심 3개 항목(근거 존재/답변 근거성/유용성)이 모두 pass면 grounded로 판정합니다.
    core_keys = ("document_relevance", "answer_grounding", "answer_helpfulness")
    core_pass = all(review[k]["status"] == "pass" for k in core_keys)
    if core_pass:
        state["grounding_status"] = "grounded"
        state["helpfulness_status"] = "helpful"
        state["needs_rewrite"] = False
        message = "Self-RAG 핵심 항목(근거 존재/답변 근거성/유용성)을 모두 만족하여 grounded로 판정했습니다."
    else:
        state["grounding_status"] = "insufficient"
        state["helpfulness_status"] = "needs_improvement"
        state["needs_rewrite"] = True
        nw = "근거 부족: Self-RAG 검증에서 fail 항목이 있어 Corrective 보정/재검색이 필요합니다."
        if nw not in state["guardrail_warnings"]:
            state["guardrail_warnings"].append(nw)
        message = "Self-RAG 검증에서 fail 항목이 있어 insufficient로 판정했습니다(Corrective 보정 필요)."

    status_map = {k: v["status"] for k, v in review.items()}
    add_trace(state, "verify_grounding_node", "success" if core_pass else "warning", message,
              f"self_rag={status_map}",
              f"grounding_status={state['grounding_status']}, warnings={len(state['guardrail_warnings'])}")
    return state


# Node 6: query_rewrite_node — 근거 부족 시 재검색 가능한 질의로 변환하는 보정 Node입니다.
def query_rewrite_node(state):
    parts = [state.get("equipment_id", ""), state.get("alarm_code", ""),
             "온도 편차", "반복 발생", "원인 후보", "1차 확인 항목"]
    rewritten = " ".join(p for p in parts if p).strip() or "온도 편차 반복 발생 원인 후보 1차 확인 항목"
    state["rewritten_query"] = rewritten
    state["retry_count"] = state.get("retry_count", 0) + 1
    state["needs_rewrite"] = False
    add_trace(state, "query_rewrite_node", "success",
              "검색 근거 부족으로 핵심 키워드 중심의 재검색 질의로 변환했습니다.",
              state.get("user_query", ""), rewritten)
    return state


# Corrective RAG 보조 함수: build_corrective_plan — 근거 부족 시 제조 Tool/MCP fallback '계획'을 만듭니다.
def build_corrective_plan(state):
    # 이 함수는 2.5 Corrective RAG의 '웹 검색 보정' 아이디어를 제조 Tool/MCP fallback으로 바꾼 것입니다.
    # - 제조/사내 환경에서는 웹 검색보다 승인된 Tool, DB/Log 조회, 품질 지표 조회, 담당자 확인 요청이 적절합니다.
    # - 실제 웹 검색이나 실제 Tool 호출은 하지 않고, 3일차 MCP Tool Contract로 확장 가능한 '계획'만 만듭니다.
    query = state.get("rewritten_query") or state.get("user_query", "") or ""
    answer = state.get("draft_answer", "") or ""
    # 질의 + 답변 초안을 함께 보고 어떤 Tool 보정이 적절한지 키워드로 판단합니다(외부 LLM 미사용).
    text = query + " " + answer

    plan = []
    recommended = []
    step = 0

    # 근거 부족(needs_rewrite/insufficient)으로 판정된 경우: 먼저 질의 재작성을 보정 1단계로 제안합니다.
    insufficient = (state.get("grounding_status") == "insufficient") or (state.get("needs_rewrite") is True)
    if insufficient:
        step += 1
        plan.append({
            "step": step,
            "action": "query_rewrite",
            "reason": "검색 근거가 부족하여 설비 ID, 알람 코드, 증상 중심으로 질의를 재작성합니다.",
        })

    # 키워드 -> 제조 Tool 매핑 규칙. 표현이 포함되면 해당 Tool을 보정 계획 후보로 추가합니다.
    # 각 Tool은 3일차 FastMCP Tool Contract로 확장될 후보입니다(여기서는 호출하지 않습니다).
    tool_rules = [
        (["매뉴얼", "조치 절차", "조치", "확인 항목", "1차 확인", "원인 후보"],
         "search_manual", "알람 코드의 매뉴얼 근거와 1차 확인 항목을 재확인합니다.",
         "3일차 search_manual MCP Tool로 확장 가능"),
        (["알람", "반복", "발생 시각", "로그", "이벤트"],
         "get_recent_alarm_events", "동일 설비/알람 코드의 최근 알람 이벤트와 반복 발생 추이를 조회합니다.",
         "3일차 get_recent_alarm_events MCP Tool로 확장 가능"),
        (["온도", "압력", "진공", "공정 상태", "공정", "챔버"],
         "get_process_status", "현재 공정/설비 상태(온도·압력·진공 등) 값을 확인합니다.",
         "3일차 get_process_status MCP Tool로 확장 가능"),
        (["품질", "불량", "수율", "품질 영향"],
         "get_quality_metrics", "해당 구간의 품질 지표(불량/수율 등) 영향 여부를 확인합니다.",
         "3일차 get_quality_metrics MCP Tool로 확장 가능"),
        (["정비", "부품", "교체", "이력", "유지보수"],
         "get_maintenance_history", "설비의 정비/부품 교체 이력을 확인합니다.",
         "3일차 get_maintenance_history MCP Tool로 확장 가능"),
    ]
    matched = []
    for keywords, tool, purpose, mcp in tool_rules:
        if any(k in text for k in keywords):
            matched.append((tool, purpose, mcp))

    # 근거가 부족한데 키워드로 잡힌 Tool이 없으면, 제조 Tool 전반을 fallback 후보로 제안합니다.
    if insufficient and not matched:
        matched = [(t, p, m) for (_ks, t, p, m) in tool_rules]

    for tool, purpose, mcp in matched:
        if tool in recommended:  # recommended_tools에는 중복 없이 Tool 이름만 남깁니다.
            continue
        step += 1
        plan.append({
            "step": step,
            "tool_name": tool,
            "purpose": purpose,
            "mcp_connection": mcp,
        })
        recommended.append(tool)

    return plan, recommended


# Node: corrective_rag_node — build_corrective_plan을 호출해 보정 계획을 State에 남기는 Corrective RAG 보정 Node입니다.
def corrective_rag_node(state):
    # 이 Node는 근거 부족 시 바로 답변을 끝내지 않고 '보정 경로'를 설계하는 단계입니다.
    # corrective_plan은 실제 Tool 호출 결과가 아니라 3일차 MCP Tool Contract로 확장할 호출 '계획'입니다.
    plan, recommended = build_corrective_plan(state)
    state["corrective_plan"] = plan

    # recommended_tools에는 중복 없이 Tool 이름만 누적합니다(3일차 MCP Tool 연결 후보).
    existing = state.get("recommended_tools", []) or []
    for t in recommended:
        if t not in existing:
            existing.append(t)
    state["recommended_tools"] = existing

    # 근거 부족 상태이므로 답변에 '추가 확인 필요 / Tool 확인 필요' 안내를 덧붙여 단정하지 않도록 합니다.
    notice = ("\n\n> [추가 확인 필요] 검색 근거가 부족하여 답변을 확정하지 않습니다. "
              "아래 Tool 확인 계획(corrective_plan)에 따라 추가 근거를 확보한 뒤 재검토가 필요합니다.")
    if notice not in (state.get("final_answer", "") or ""):
        state["final_answer"] = (state.get("final_answer", "") or "") + notice
        state["draft_answer"] = (state.get("draft_answer", "") or "") + notice

    # trace에 보정 계획 생성 결과를 남겨 4일차 실행 품질 평가에서 검토할 수 있게 합니다.
    add_trace(state, "corrective_rag_node", "warning",
              "근거 부족으로 제조 Tool/MCP fallback 보정 계획을 생성했습니다(실제 호출 아님).",
              f"insufficient grounding (recommended={len(recommended)})",
              f"corrective_plan={len(plan)}단계, recommended_tools={len(state['recommended_tools'])}개")
    return state


print("Node/검증/보정 함수 정의 완료:",
      "parse_query_node, retrieve_alarm_logs_node, retrieve_docs_node, generate_answer_node,",
      "verify_grounding_node(+run_self_rag_review), query_rewrite_node, corrective_rag_node(+build_corrective_plan)")

## 6. 조건부 분기 함수 정의

조건부 분기는 **근거 없는 답변을 막기 위한 품질 관리 구조**이자, 무한 재작성을 막는 안전 장치입니다.
Self-RAG식 검증에서 근거가 부족하면 **Corrective RAG식 보정 경로(`corrective_rag_node`)** 를 명시적으로 지나가도록 분기합니다.

- `grounding_status == "grounded"` → `END`
- 근거 부족(`needs_rewrite == True`)이고 `retry_count <= 1` → `corrective_rag_node`(제조 Tool/MCP 보정 계획 생성) → `query_rewrite_node` → `retrieve_docs_node` 재검색
- 재시도 한도(`retry_count > 1`)를 초과하면 → `END`

> 분기 함수는 문자열(`"end"` / `"corrective"`)을 반환하고, `add_conditional_edges`에서 `"end" → END`, `"corrective" → corrective_rag_node`로 매핑합니다.

In [ ]:
# verify_grounding_node 이후 라우팅: 근거 충분/재시도 한도에 따라 종료 또는 Corrective 보정으로 분기
def decide_after_grounding(state):
    # 근거가 충분하면(grounded) 종료합니다.
    if state.get("grounding_status") == "grounded":
        return "end"
    # 재검색 루프는 최대 1회만 허용합니다. retry_count가 1을 초과하면 무한 루프 방지를 위해 종료합니다.
    if state.get("retry_count", 0) > 1:
        return "end"
    # 근거가 부족하면(needs_rewrite=True) Corrective RAG 보정 경로로 명시적으로 보냅니다.
    if state.get("needs_rewrite") is True:
        return "corrective"
    # 그 외에는 안전하게 종료합니다.
    return "end"


print("조건부 분기 함수 정의 완료: decide_after_grounding (반환값: 'end' / 'corrective')")

## 7. StateGraph 구성

실제 **LangGraph StateGraph**로 그래프를 구성합니다.
근거가 부족하면 **Corrective RAG식 보정 Node(`corrective_rag_node`)** 를 지나 질의를 재작성하고 다시 검색하는 보정 루프를 만듭니다.

```
parse_query_node → retrieve_alarm_logs_node → retrieve_docs_node → generate_answer_node → verify_grounding_node → 조건부 분기
   ├─ grounded: END
   └─ insufficient: corrective_rag_node → query_rewrite_node → retrieve_docs_node (재검색, 최대 1회)
```

> LangGraph import/구성이 실패하면 `graph = None`으로 두고, 다음 셀에서 fallback final_state·trace를 생성해 흐름을 유지합니다.

In [ ]:
# 이 셀은 위에서 정의한 Node/검증/보정 함수를 실제 LangGraph StateGraph로 연결하는 단계입니다.
# - StateGraph는 Node(실행 단위)와 Edge(이동 경로)를 명시적으로 연결하는 LangGraph의 핵심 구조입니다.
# - 근거가 부족하면 corrective_rag_node를 거쳐 query_rewrite_node -> retrieve_docs_node로 돌아가는 보정 루프를 만듭니다.
# - LangGraph import/구성이 실패해도 graph=None으로 두고 다음 셀의 fallback 흐름으로 강의를 이어갑니다.
graph = None
try:
    from langgraph.graph import StateGraph, END

    builder = StateGraph(AgentState)
    # add_node: State를 입력받아 갱신하는 실행 단위(Node)를 그래프에 등록합니다.
    builder.add_node("parse_query_node", parse_query_node)
    builder.add_node("retrieve_alarm_logs_node", retrieve_alarm_logs_node)
    builder.add_node("retrieve_docs_node", retrieve_docs_node)
    builder.add_node("generate_answer_node", generate_answer_node)
    builder.add_node("verify_grounding_node", verify_grounding_node)
    # corrective_rag_node: 근거 부족 시 제조 Tool/MCP fallback 계획을 생성하는 보정 Node입니다(실제 호출 아님).
    builder.add_node("corrective_rag_node", corrective_rag_node)
    builder.add_node("query_rewrite_node", query_rewrite_node)

    # set_entry_point / add_edge: 그래프의 시작점과 고정된 실행 순서를 정의합니다.
    builder.set_entry_point("parse_query_node")
    builder.add_edge("parse_query_node", "retrieve_alarm_logs_node")
    builder.add_edge("retrieve_alarm_logs_node", "retrieve_docs_node")
    builder.add_edge("retrieve_docs_node", "generate_answer_node")
    builder.add_edge("generate_answer_node", "verify_grounding_node")
    # add_conditional_edges: State 값(grounding_status/needs_rewrite/retry_count)에 따라 다음 Node를 다르게 선택합니다.
    # grounded면 END, 근거 부족이면 corrective_rag_node로 보내 보정 계획을 만든 뒤 재검색 루프로 진입합니다.
    builder.add_conditional_edges("verify_grounding_node", decide_after_grounding,
                                  {"corrective": "corrective_rag_node", "end": END})
    # Corrective 보정 -> 질의 재작성 -> 문서 재검색으로 이어지는 Corrective RAG식 보정 루프입니다.
    builder.add_edge("corrective_rag_node", "query_rewrite_node")
    builder.add_edge("query_rewrite_node", "retrieve_docs_node")

    # compile: 설계한 Node/Edge 구성을 실행 가능한 Graph 객체로 만드는 단계입니다.
    graph = builder.compile()
    print("LangGraph StateGraph 컴파일 완료 (graph 생성됨, corrective_rag_node 포함)")
except Exception as exc:
    # LangGraph가 없거나 구성에 실패하면 graph=None으로 두어 노트북이 중단되지 않게 합니다.
    graph = None
    display(Markdown(
        f"> LangGraph를 사용할 수 없어 `graph=None`으로 둡니다(`{type(exc).__name__}`). "
        f"다음 셀에서 fallback final_state와 fallback trace로 노트북 실행 흐름을 유지합니다."
    ))

## 8. graph.invoke 실행과 Trace 저장

`graph`가 정상 생성되었으면 `graph.invoke(initial_state)`로 그래프를 실행합니다.
실행이 실패해도 노트북이 중단되지 않도록 **fallback final_state**를 생성하며, `final_state` 변수는 반드시 남깁니다.
Trace는 `outputs/day2/day2_langgraph_trace.md`로 저장합니다.

In [ ]:
# 이 셀은 graph.invoke(initial_state)로 Graph를 실행하고 결과를 final_state에 담는 셀입니다.
# - final_state에는 검색 근거, 답변, Self-RAG 검증 결과, Corrective RAG 보정 계획, Trace가 모두 포함됩니다.
# - graph.invoke가 실패하거나 LangGraph가 없어도 강의 흐름이 멈추지 않도록 fallback final_state를 생성합니다.
# - Node 실행 순서와 State 변화는 day2_langgraph_trace.md 산출물로 저장합니다.
import copy


def run_fallback_pipeline(state, note):
    # fallback final_state는 LangGraph 환경/State 타입 문제로 invoke가 실패해도 이후 흐름을 유지하기 위한 안전 장치입니다.
    parse_query_node(state)
    retrieve_alarm_logs_node(state)
    retrieve_docs_node(state)
    generate_answer_node(state)
    verify_grounding_node(state)
    # 근거 부족이면 Corrective RAG 보정 경로(corrective_rag_node -> query_rewrite_node -> 재검색)를 한 번 수행합니다.
    if decide_after_grounding(state) == "corrective":
        corrective_rag_node(state)
        query_rewrite_node(state)
        retrieve_docs_node(state)
        generate_answer_node(state)
        verify_grounding_node(state)
    state["errors"].append(note)
    add_trace(state, "fallback", "warning", note, state.get("user_query", ""),
              f"grounding_status={state.get('grounding_status', '')}")
    return state


if graph is not None:
    try:
        final_state = graph.invoke(copy.deepcopy(INITIAL_STATE))
    except Exception as exc:
        display(Markdown(f"> graph.invoke 실패(`{type(exc).__name__}`) → fallback final_state로 진행합니다."))
        final_state = run_fallback_pipeline(copy.deepcopy(INITIAL_STATE), f"graph.invoke 실패: {type(exc).__name__}")
else:
    final_state = run_fallback_pipeline(copy.deepcopy(INITIAL_STATE), "LangGraph 미설치/임포트 실패로 fallback trace를 생성했습니다.")

# Trace 산출물 저장
tlines = ["# Day2 LangGraph 실행 Trace", "",
          f"- LangGraph 사용: {graph is not None}", "",
          "## 1. 사용자 질문", "", f"- {final_state.get('user_query', '')}", "",
          "## 2. 최종 State 요약", "",
          f"- equipment_id: {final_state.get('equipment_id', '')}",
          f"- alarm_code: {final_state.get('alarm_code', '')}",
          f"- grounding_status: {final_state.get('grounding_status', '')}",
          f"- helpfulness_status: {final_state.get('helpfulness_status', '')}",
          f"- needs_rewrite: {final_state.get('needs_rewrite', False)}",
          f"- retry_count: {final_state.get('retry_count', 0)}",
          f"- retrieved_logs 수: {len(final_state.get('retrieved_logs', []))}",
          f"- retrieved_docs 수: {len(final_state.get('retrieved_docs', []))}", "",
          "## 3. Node 실행 Trace", "",
          "| 순서 | node_name | status | output_summary |", "|---:|---|---|---|"]
for i, tr in enumerate(final_state.get("trace", []), 1):
    tlines.append(f"| {i} | {tr.get('node_name', '')} | {tr.get('status', '')} | {esc(tr.get('output_summary', ''))} |")
tlines.append("")
save_markdown("outputs/day2/day2_langgraph_trace.md", "\n".join(tlines))

print("grounding_status:", final_state.get("grounding_status", ""))
print("Node 실행 순서:", [t.get("node_name", "") for t in final_state.get("trace", [])])

## LangGraph 구조 시각화

- 이 셀은 실제 LangGraph로 구성한 RAG Agent v1의 Node 흐름을 시각적으로 확인하는 **선택 시각화 셀**입니다.
- Node 실행 로직은 이전 셀에서 수행합니다(시각화 실패가 `graph.invoke` 흐름에 영향을 주지 않도록 별도 셀로 분리).
- `graph` 객체가 정상 생성되었을 때 **Mermaid PNG**를 출력하며, 렌더링 환경 문제로 실패하면 **Markdown 흐름도**로 대체합니다.

In [ ]:
# LangGraph 구조 시각화 (선택). Node 실행 셀과 분리된 별도 셀입니다.
# - Node는 State를 입력받아 갱신하는 Agent 실행 단위이고, 조건부 분기는 근거 없는 답변을 막기 위한 품질 관리 구조입니다.
# - LangGraph 렌더링 환경이 없으면 Markdown 흐름도로 대체합니다.
# - Mermaid PNG 생성 실패는 코드 오류라기보다 렌더링 환경 문제일 수 있습니다.
from IPython.display import Image, Markdown, display

_fallback_flow = """**LangGraph 구조 (Markdown 흐름도)**

```
parse_query_node
→ retrieve_alarm_logs_node
→ retrieve_docs_node
→ generate_answer_node
→ verify_grounding_node
→ 조건부 분기
   ├─ grounding 충분: END
   └─ grounding 부족: corrective_rag_node → query_rewrite_node → retrieve_docs_node 재검색
```
"""

graph_obj = globals().get("graph", None)
try:
    if graph_obj is not None and hasattr(graph_obj, "get_graph"):
        display(Image(graph_obj.get_graph().draw_mermaid_png()))
    else:
        display(Markdown("graph 객체가 없어 Markdown 흐름도로 대체합니다."))
        display(Markdown(_fallback_flow))
except Exception as exc:
    display(Markdown(f"LangGraph 그래프 이미지 출력 중 문제가 발생하여 Markdown 흐름도로 대체합니다. (`{type(exc).__name__}`)"))
    display(Markdown(_fallback_flow))

## final_state와 Trace 해석

Trace는 **Node 실행 순서와 State 변화를 확인하는 실행 검토 자료**이며, **4일차 실행 품질 평가의 기반**입니다.
아래 셀에서 최종 State의 주요 필드와 Trace 단계를 보기 좋게 확인합니다.

In [ ]:
# 이 셀은 Graph 실행이 끝난 final_state를 해석하는 셀입니다.
# - final_state는 검색 근거, 답변, Self-RAG 검증 결과, Corrective RAG 보정 계획, Trace를 모두 담은 최종 State입니다.
# - 화면에는 전체 JSON 대신 핵심 필드만 요약해서 보여주고, 상세는 산출물(Markdown)로 확인합니다.
# - Self-RAG/Corrective RAG 보강 필드는 없을 수도 있으므로 빈 dict/list로 안전하게 처리합니다.
fa = final_state.get("final_answer", "") or ""
self_rag = final_state.get("self_rag_review", {}) or {}
corrective_plan = final_state.get("corrective_plan", []) or []
recommended_tools = final_state.get("recommended_tools", []) or []
guardrail_warnings = final_state.get("guardrail_warnings", []) or []

# 핵심 필드 요약: 원본 요청·검색 조건, 근거 확보 여부, Self-RAG 검증 결과, Corrective RAG 보정 계획을 한눈에 봅니다.
summary = {
    "user_query": final_state.get("user_query", ""),
    "equipment_id": final_state.get("equipment_id", ""),
    "alarm_code": final_state.get("alarm_code", ""),
    "grounding_status": final_state.get("grounding_status", ""),      # Self-RAG식 검증 결과(grounded/insufficient)
    "helpfulness_status": final_state.get("helpfulness_status", ""),
    "needs_rewrite": final_state.get("needs_rewrite", False),
    "retry_count": final_state.get("retry_count", 0),
    "retrieved_logs 개수": len(final_state.get("retrieved_logs", [])),  # 근거 확보 여부 확인 기준
    "retrieved_docs 개수": len(final_state.get("retrieved_docs", [])),
    # Self-RAG 검증은 항목별 status만 요약해서 표시합니다(전체 JSON은 길어서 생략).
    "self_rag_review(상태 요약)": {k: v.get("status", "") for k, v in self_rag.items()},
    "corrective_plan 단계 수": len(corrective_plan),                    # Corrective RAG 보정 계획 단계 수
    "recommended_tools": recommended_tools,                            # 3일차 MCP Tool 연결 후보
    "guardrail_warnings 개수": len(guardrail_warnings),                 # 단정/근거 부족/민감정보 가능성 경고 수
    "final_answer 일부": fa[:160] + ("..." if len(fa) > 160 else ""),
}
display(summary)

# trace는 Node 실행 순서와 주요 판단 근거를 보여주며, 4일차 실행 품질 평가의 기준 자료가 됩니다.
print("\ntrace 단계 목록:")
for i, tr in enumerate(final_state.get("trace", []), 1):
    print(f"  {i}. {tr.get('node_name', '')} [{tr.get('status', '')}] - {tr.get('output_summary', '')}")

# guardrail_warnings는 과도한 단정·근거 부족·민감정보 가능성을 점검한 결과로, 4일차 Guardrail/평가의 입력이 됩니다.
print("\nguardrail_warnings:")
if guardrail_warnings:
    for w in guardrail_warnings:
        print("  -", w)
else:
    print("  - (경고 없음)")

# 이미 저장한 Trace 산출물(day2_langgraph_trace.md)을 노트북에서 함께 렌더링해 실행 흐름을 검토합니다.
print()
show_markdown("outputs/day2/day2_langgraph_trace.md")

## Self-RAG 검증 결과와 Corrective RAG 보정 계획 산출물

위 final_state의 **Self-RAG 검증 결과(`self_rag_review`)** 와 **Corrective RAG 보정 계획(`corrective_plan`)** 을
`outputs/day2/day2_langgraph_self_corrective_review.md` 산출물로 저장합니다(추가 산출물, 기존 trace.md는 유지).

- 검증은 외부 LLM grader 없이 **규칙 기반**으로 수행했습니다.
- 보정 계획은 웹 검색이 아니라 **제조 Tool/MCP fallback 계획**이며, 실제 Tool 호출은 하지 않습니다.
- 이번 실행이 `grounded`로 끝나 그래프에서 `corrective_rag_node`가 실행되지 않았다면, 교육 목적상 **"근거가 부족했다면 어떤 보정 계획이 가능한지"** 를 함께 계산해 산출물에 보여줍니다.

In [ ]:
# 이 셀은 Self-RAG 검증 결과와 Corrective RAG 보정 계획을 Markdown 산출물로 저장하는 셀입니다.
# - day2_langgraph_self_corrective_review.md는 답변의 근거성/유용성/안전성(Self-RAG)과
#   근거 부족 시 어떤 Tool/MCP 흐름으로 확장할지(Corrective RAG)를 강의 중 함께 확인하기 위한 산출물입니다.
# - 기존 day2_langgraph_trace.md는 그대로 두고, 추가 산출물만 생성합니다.
review = final_state.get("self_rag_review", {}) or {}
warnings = final_state.get("guardrail_warnings", []) or []

# 그래프가 grounded로 끝나면 corrective_rag_node가 실행되지 않아 corrective_plan이 비어 있을 수 있습니다.
# 교육 목적상 '근거가 부족했다면 어떤 보정 계획이 가능한지'를 함께 보여주기 위해 여기서 계획을 계산합니다(실제 호출 아님).
plan = final_state.get("corrective_plan", []) or []
recommended = final_state.get("recommended_tools", []) or []
if not plan:
    plan, recommended = build_corrective_plan(final_state)

label_map = {
    "document_relevance": "문서 관련성 (document_relevance)",
    "answer_grounding": "답변 근거성 (answer_grounding)",
    "answer_helpfulness": "답변 유용성 (answer_helpfulness)",
    "overclaim_check": "단정 표현 점검 (overclaim_check)",
    "safety_check": "민감정보 점검 (safety_check)",
}

md = []
md.append("# Day2 LangGraph Self-RAG / Corrective RAG 검토 산출물")
md.append("")
md.append("> 이 산출물은 외부 LLM grader/웹 검색 없이 **규칙 기반 검증**과 **제조 Tool/MCP fallback 계획**으로 작성되었습니다(실제 Tool 호출 없음).")
md.append("")
md.append("## 1. 사용자 질문")
md.append("")
md.append(f"- 질문: {esc(final_state.get('user_query', ''))}")
md.append(f"- equipment_id: {final_state.get('equipment_id', '') or '미확인'}")
md.append(f"- alarm_code: {final_state.get('alarm_code', '') or '미확인'}")
md.append(f"- grounding_status: {final_state.get('grounding_status', '')}")
md.append("")
# Self-RAG 검증 결과 표: 답변의 근거성·유용성·안전성을 항목별로 검토합니다.
md.append("## 2. Self-RAG 검증 결과")
md.append("")
md.append("| 검증 항목 | 상태 | 판단 이유 |")
md.append("|---|---|---|")
for key in ("document_relevance", "answer_grounding", "answer_helpfulness", "overclaim_check", "safety_check"):
    item = review.get(key, {}) or {}
    md.append(f"| {label_map.get(key, key)} | {item.get('status', '-')} | {esc(item.get('reason', ''))} |")
md.append("")
md.append("## 3. Guardrail 경고 (guardrail_warnings)")
md.append("")
if warnings:
    for w in warnings:
        md.append(f"- {esc(w)}")
else:
    md.append("- (경고 없음) 원인 단정·근거 부족·민감정보 가능성 경고가 발견되지 않았습니다.")
md.append("")
# Corrective RAG 보정 계획 표: 근거 부족 시 어떤 Tool/MCP 흐름으로 확장할 수 있는지 단계별로 보여줍니다.
md.append("## 4. Corrective RAG 보정 계획 (제조 Tool/MCP fallback)")
md.append("")
md.append("> 원본 Corrective RAG는 근거 부족 시 웹 검색으로 보정하지만, 제조/사내 환경에서는 승인된 Tool과 제한된 데이터 소스로 보정합니다. 아래는 실제 호출이 아니라 3일차 MCP Tool로 확장 가능한 계획입니다.")
md.append("")
md.append("| 단계 | Action/Tool | 목적 | MCP 연결 |")
md.append("|---:|---|---|---|")
for s in plan:
    action = s.get("tool_name") or s.get("action", "")
    purpose = s.get("purpose") or s.get("reason", "")
    mcp = s.get("mcp_connection", "-")
    md.append(f"| {s.get('step', '')} | {esc(action)} | {esc(purpose)} | {esc(mcp)} |")
md.append("")
# recommended_tools: 3일차 MCP Tool 연결 후보(중복 없는 Tool 이름 목록).
md.append("## 5. 추천 Tool 후보 (recommended_tools)")
md.append("")
if recommended:
    for t in recommended:
        md.append(f"- {esc(t)}")
else:
    md.append("- (없음)")
md.append("")
md.append("## 6. 3일차 MCP Tool 연결")
md.append("")
md.append("- 위 보정 계획의 Tool 후보는 3일차에 FastMCP 기반 Tool Contract(search_manual, get_recent_alarm_events, get_process_status, get_quality_metrics, get_maintenance_history)로 확장됩니다.")
md.append("- 이 노트북에서는 실제 호출 없이 '어떤 Tool을 어떤 목적으로 호출할지'에 대한 계획만 State(corrective_plan)에 남깁니다.")
md.append("")
md.append("## 7. 4일차 Guardrail/Trace 평가 연결")
md.append("")
md.append("- self_rag_review와 guardrail_warnings는 4일차의 Guardrail/품질 게이트, Trace 평가 기준으로 확장됩니다.")
md.append("- trace와 함께 보면 '어떤 Node에서 근거가 부족했고, 어떤 보정 계획을 세웠는지'를 실행 품질 관점에서 평가할 수 있습니다.")
md.append("")

# 추가 산출물 저장(outputs/day2)과 노트북 내 렌더링. 기존 trace.md는 그대로 유지합니다.
save_markdown("outputs/day2/day2_langgraph_self_corrective_review.md", "\n".join(md))
show_markdown("outputs/day2/day2_langgraph_self_corrective_review.md")

## Self-RAG식 검증 구조 설명

2.4 Self-RAG 예제의 핵심은 **생성된 답변을 여러 관점에서 검증**하는 것입니다.

이 노트북은 **외부 LLM grader를 사용하지 않고**, `verify_grounding_node`가 호출하는 `run_self_rag_review` 함수에서 **규칙 기반으로 축소 적용**했습니다.

**검증 항목 (규칙 기반)**

| 항목 | 점검 내용 | 결과 |
|---|---|---|
| `document_relevance` | retrieved_docs / retrieved_logs 근거 존재 여부 | pass / fail |
| `answer_grounding` | 답변이 설비 ID·알람 코드·근거 키워드를 반영하는가 | pass / fail |
| `answer_helpfulness` | 확인 / 점검 / 검토 / 조치 / 추가 확인 등 실무 안내 포함 여부 | pass / fail |
| `overclaim_check` | "확정 원인 / 반드시 / 즉시 조치 / 무조건" 등 단정 표현 여부 | pass / warning |
| `safety_check` | 사번·연락처 등 민감정보, 실제 사내 데이터처럼 보이는 표현 여부 | pass / warning |

- 핵심 3개 항목(`document_relevance`, `answer_grounding`, `answer_helpfulness`)이 모두 `pass`이면 `grounded`로 판정합니다.
- 하나라도 `fail`이면 `insufficient` → 재작성/보정이 필요한 상태로 둡니다.
- `warning`은 `guardrail_warnings`에 누적되어 안전 점검 근거로 남습니다.

이 구조는 **4일차 Guardrail/평가 구조**로 확장할 수 있으며, 실제 운영에서는 평가 모델, 정책 규칙, 내부 검증 절차와 결합할 수 있습니다.

## Corrective RAG 개념을 제조 Tool fallback으로 연결

2.5 Corrective RAG는 검색 근거가 부족할 때 **외부 웹 검색으로 보정**하는 구조입니다.
그러나 제조/사내 환경에서는 웹 검색을 그대로 쓰기보다 **승인된 Tool과 제한된 데이터 소스로 보정 흐름을 확장**하는 것이 적절합니다.

이 노트북은 Corrective RAG를 **설명만 하지 않고**, 근거 부족 시 `corrective_rag_node`가 호출하는 `build_corrective_plan` 함수로
**제조 Tool/MCP fallback 보정 계획(`corrective_plan`)** 을 실제 코드로 생성합니다. (실제 Tool 호출이나 웹 검색은 하지 않습니다.)

**보정 계획에 제안되는 Tool 후보**

- `search_manual` — 알람 코드 매뉴얼 근거 / 1차 확인 항목 재확인
- `get_recent_alarm_events` — 최근 알람 이벤트·반복 발생 추이 조회
- `get_process_status` — 온도·압력·진공 등 공정/설비 상태 확인
- `get_quality_metrics` — 품질 지표(불량/수율 등) 영향 여부 확인
- `get_maintenance_history` — 정비·부품 교체 이력 확인
- (필요 시) 담당자 확인 요청

> Tavily / DuckDuckGo 등 외부 Web Search Tool은 사용하지 않으며, 웹 검색 코드도 구현하지 않습니다.

**핵심 메시지**
Corrective RAG의 핵심은 웹 검색 자체가 아니라, **근거 부족 시 보정 경로를 설계**하는 것입니다.
이 노트북에서는 `verify_grounding_node`(Self-RAG식 검증) → `corrective_rag_node`(보정 계획 생성) → `query_rewrite_node` → `retrieve_docs_node` 재검색 흐름으로 그 보정 경로를 구현하며,
생성된 계획은 실제 호출 없이 **3일차 MCP Tool Contract로 확장 가능한 계획**으로만 남습니다.

## 3일차 MCP Tool 연결

오늘 설계한 Node/검증/보정 흐름은 3일차에 **MCP Tool 호출 구조**로 확장됩니다.

| 2일차 LangGraph 구조 | 3일차 이후 확장 |
|---|---|
| `retrieve_docs_node` | → `search_manual` Tool |
| `retrieve_alarm_logs_node` | → `get_recent_alarm_events` Tool |
| `verify_grounding_node` | → Guardrail / 평가 Node |
| `query_rewrite_node` | → 재검색 또는 Tool 재호출 조건 |
| `trace` | → MCP Call Trace 및 4일차 실행 품질 평가 |